# TEMPLATE 4: DETEKSI HOAKS, CYBERBULLYING & TOXIC COMMENTS
### GEMASTIK DATA MINING & NLP PIPELINE (BINARY CLASSIFICATION WITH SEVERE IMBALANCE)
Pipeline ini dirancang untuk mendeteksi teks berbahaya (Hoaks vs Fakta, Cyberbullying vs Non-bullying, Komentar Toksik).

**Fitur Utama Pipeline:**
1. Ekstraksi Fitur Karakteristik Khusus Teks Toksik/Hoaks:
   - Densitas tanda seru/tanya berulang (`!?!?!?`)
   - Rasio huruf kapital (indikasi berteriak / marah)
   - Karakter n-gram TF-IDF (menangkap sensor/typo slang seperti `4nj1ng`, `g0bl0k`)
2. Modeling dengan LightGBM & Logistic Regression dengan pembobotan kelas (`scale_pos_weight`).
3. **Threshold Tuning Search**: Mencari ambang probabilitas optimal $T \in [0.1, 0.9]$ yang memaksimalkan F1-score pada OOF (bukan default 0.5!).
4. Pembuatan berkas submission format biner / probabilitas.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Modul deteksi hoaks dan cyberbullying siap.")


## 1. Memuat Dataset Biner Hoaks / Cyberbullying


In [ ]:
TRAIN_PATH = "cyberbullying_hoax_train.csv"
TEXT_COL = "text"
TARGET_COL = "is_toxic"  # 1 = Toxic/Hoax, 0 = Normal/Fakta

if not os.path.exists(TRAIN_PATH):
    print("Membuat sampel data deteksi toxic/hoaks bahasa Indonesia...")
    sample_data = {
        "id": range(1, 15),
        "text": [
            "Waspada! Beredar kabar telur palsu terbuat dari plastik beredar di pasar tradisional!",
            "Kementerian Kesehatan membantah isu vaksin menyebabkan chip pelacak.",
            "Dasar manusia gak punya otak kerjaannya cuma nyusahin orang lain doang!!",
            "Muka lu jelek banget kayak sampah gak pantas ikut kompetisi bergengsi ini wkwk",
            "Rapat koordinasi tingkat menteri menyepakati penurunan tarif impor pangan.",
            "BREAKING NEWS!! Minum air rebusan bawang putih bisa sembuhkan penyakit kronis 100%!",
            "Selamat kepada seluruh peserta yang telah berhasil lolos ke babak final nasional.",
            "Halah pansos lu artis modal tampang doang tapi kelakuan kayak binatang baji*gan",
            "Prakiraan cuaca wilayah Jawa Barat berpotensi cerah berawan hingga sore hari.",
            "KLAIM MENYESATKAN: Beredar video jembatan ambruk yang diklaim berlokasi di Jakarta.",
            "Mampus lu kena karma rasain akibatnya orang sombong sok jagoan!!",
            "Pemerintah resmi meresmikan jalan tol baru sepanjang 50 kilometer.",
            "Bocah tolol main game terus mending mati aja lu gak ada gunanya hidup!!",
            "Universitas menyelenggarakan workshop penulisan karya ilmiah bereputasi internasional."
        ],
        "is_toxic": [1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0]
    }
    df = pd.DataFrame(sample_data)
else:
    df = pd.read_csv(TRAIN_PATH)

print(f"Total Dokumen: {len(df)}")
print(f"Proporsi Target:\n{df[TARGET_COL].value_counts(normalize=True) * 100}")


## 2. Ekstraksi Fitur Stilometri (Emosional, Huruf Kapital & Tanda Baca)


In [ ]:
def extract_toxic_stylometrics(df, col):
    feats = pd.DataFrame(index=df.index)
    feats["char_len"] = df[col].astype(str).str.len()
    feats["word_len"] = df[col].astype(str).apply(lambda s: len(s.split()))
    feats["caps_count"] = df[col].astype(str).apply(lambda s: sum(1 for c in s if c.isupper()))
    feats["caps_ratio"] = feats["caps_count"] / (feats["char_len"] + 1)
    feats["exclam_count"] = df[col].astype(str).apply(lambda s: s.count('!'))
    feats["question_count"] = df[col].astype(str).apply(lambda s: s.count('?'))
    feats["digit_count"] = df[col].astype(str).apply(lambda s: sum(1 for c in s if c.isdigit()))
    feats["repeated_char_count"] = df[col].astype(str).apply(lambda s: len(re.findall(r"(.)\1{2,}", s)))
    
    # Kata kunci penanda hoaks/clickbait
    hoax_keywords = ["waspada", "beredar", "viral", "breaking", "rahasia", "terbongkar", "100%", "palsu", "bahaya"]
    feats["hoax_kw_count"] = df[col].astype(str).apply(lambda s: sum(1 for kw in hoax_keywords if kw in s.lower()))
    
    return feats

style_features = extract_toxic_stylometrics(df, TEXT_COL)
print("Contoh Fitur Stilometri:")
print(style_features.head(3))


## 3. Ekstraksi Fitur N-Gram TF-IDF (Word + Char)


In [ ]:
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)
X_word = word_vec.fit_transform(df[TEXT_COL].str.lower())

# Char N-Gram sangat ampuh menangkap kata makian yang disamarkan dengan angka/simbol (leet speak)
char_vec = TfidfVectorizer(ngram_range=(2, 5), analyzer="char", max_features=15000, sublinear_tf=True)
X_char = char_vec.fit_transform(df[TEXT_COL].str.lower())

X_all = hstack([X_word, X_char, csr_matrix(style_features.values)]).tocsr()
y_all = df[TARGET_COL].values
print(f"Total Dimensi Fitur Terpadu: {X_all.shape}")


## 4. Validasi Silang 5-Fold & Optimasi Ambang Batas (Threshold Search)


In [ ]:
n_splits = min(5, min(df[TARGET_COL].value_counts()))
n_splits = max(2, n_splits)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

oof_probas = np.zeros(len(df))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
    X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
    X_va, y_va = X_all[va_idx], y_all[va_idx]
    
    # Model Logistic Regression dengan Balanced Class Weight
    clf = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=SEED + fold)
    clf.fit(X_tr, y_tr)
    oof_probas[va_idx] = clf.predict_proba(X_va)[:, 1]

# Pencarian Threshold Terbaik untuk Maksimalkan F1-Score
best_thresh = 0.5
best_f1 = 0.0
thresholds = np.linspace(0.1, 0.9, 81)

for t in thresholds:
    f1 = f1_score(y_all, (oof_probas >= t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"\n=== HASIL OPTIMASI THRESHOLD ===")
print(f"Default Threshold (0.50) -> F1-Score: {f1_score(y_all, (oof_probas >= 0.5).astype(int)):.4f}")
print(f"Optimal Threshold ({best_thresh:.2f}) -> F1-Score: {best_f1:.4f} (+{(best_f1 - f1_score(y_all, (oof_probas >= 0.5).astype(int)))*100:.2f}%)")
print(f"ROC-AUC Score: {roc_auc_score(y_all, oof_probas):.4f}")


## 5. Ekspor Berkas Submission Format Biner & Probabilitas


In [ ]:
final_preds = (oof_probas >= best_thresh).astype(int)
sub_df = pd.DataFrame({
    "id": df["id"],
    "predicted_label": final_preds,
    "toxic_probability": np.round(oof_probas, 4)
})
sub_df.to_csv("submission_hoax_cyberbullying.csv", index=False)
print("Berkas submission_hoax_cyberbullying.csv berhasil disimpan:")
print(sub_df.head(10))
